# 📋 [OPS08_M01] 야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m01_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M01
- **미션 주제:** 야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 01. 야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"야전 경계진지 드론 출현 가상 사례 C1~C6(정찰, 자폭, 통신두절 등)의 맥락을 분석하여, 최초 포착부터 전파 지연까지 발생한 핵심 전술적 문제점을 1줄 요약 표로 정리해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - AI가 군사 보안 규정을 준수하며, 개인 실수가 아닌 7단계 지휘통제 절차 결함 관점으로 분석했는지 검증합니다.

---

### 📑 실습자료 및 과업 지침

야전 부대 드론 경보 상황 분석 및 관측자-지휘자-경계조 역할 정의**

가. 제공 자료 - 가상 드론 출현 사례 카드 [교육용가상자료]

| **카드** | **가상 시각·기상** | **상황** | **드러난 문제** |
| --- | --- | --- | --- |
| **C1** | 05:42, 옅은 안개 | OP-1 관측자가 윙윙거리는 소리를 들었으나 방향을 특정하지 못하고 '뭔가 소리가 난다'고만 보고했다. | 방위 없는 보고, 지휘자 되묻기 2회로 보고 지연 약 2분 |
| **C2** | 11:10, 맑음 | 12시 방향 나무 높이 위에서 드론 1기가 선회했다. 관측자는 즉시 보고했으나 경계조가 대피 위치를 몰라 우왕좌왕했다. | 대피 위치 사전 지정 없음 |
| **C3** | 14:25, 흐림 | 조류 무리를 드론으로 오인해 경보를 발령했고 경계조 전원이 대피해 5분간 경계 공백이 생겼다. | 교차 확인 없이 경보, 전원 대피로 경계 공백 |
| **C4** | 19:50, 해질녘 | 드론 2기가 접근했으나 관측자는 1기만 보고했다. 두 번째 기체는 경계조가 봤지만 누구에게 보고할지 몰랐다. | 경계조의 보고 경로 불명확 |
| **C5** | 23:30, 야간 | 무전 잡음 속에서 방위 '3시'가 '9시'로 전달되어 반대 방향을 경계했다. | 복창 확인 절차 없음 |
| **C6** | 02:15, 야간 | 지휘자가 순찰 중이라 부재했고 차선임자가 결심해도 되는지 몰라 대응이 지연되었다. | 지휘 위임 순서 미지정 |

나. 작성 서식 1 - 역할 분담 매트릭스 (수: 수행, 책: 책임, 협: 협조, 통: 통보 / 단계마다 '책'은 1명)

| **단계** | **관측자** | **지휘자** | **경계조** | **통신·기록** | **완료 판단 기준** |
| --- | --- | --- | --- | --- | --- |
| **① 포착** | 책·수 (예시) | 통 (예시) | 협 (예시) | - | 포착 시각 기록 (예시) |
| **② 식별** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **③ 보고** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **④ 전파** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **⑤ 결심** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **⑥ 대피·대응** | ( ) | ( ) | ( ) | ( ) | ( ) |
| **⑦ 해제** | ( ) | ( ) | ( ) | ( ) | ( ) |

다. 작성 서식 2 - 현장 체크 항목 정의표 (15개 이상)

| **번호** | **단계** | **체크 항목(눈으로 확인 가능한 문장)** | **확인 방법** | **담당** |
| --- | --- | --- | --- | --- |
| **1** | 포착 | 관측자가 담당 관측 구역을 말로 설명할 수 있다 (예시) | 평가관 질문 (예시) | 관측자 (예시) |
| **2** | 보고 | 1차 보고에 방위가 포함되었다 (예시) | 기록표 확인 (예시) | 지휘자 (예시) |
| **3** |  |  |  |  |
| **4** |  |  |  |  |
| **5** |  |  |  |  |
| **6** |  |  |  |  |
| **7** |  |  |  |  |
| **8** |  |  |  |  |
| **9** |  |  |  |  |
| **10** |  |  |  |  |
| **11** |  |  |  |  |
| **12** |  |  |  |  |
| **13** |  |  |  |  |
| **14** |  |  |  |  |
| **15** |  |  |  |  |
| **16** |  |  |  |  |
| **17** |  |  |  |  |

라. 작성 서식 3 - 부대 특성별 환경 변수 보강표

| **환경 변수** | **야전 진지** | **기지(주둔지)** | **의무시설** |
| --- | --- | --- | --- |
| **관측 시야(차폐물)** | ( ) | ( ) | ( ) |
| **대피·은폐 장소** | ( ) | ( ) | ( ) |
| **통신 수단** | ( ) | ( ) | ( ) |
| **경보 전파 대상** | ( ) | ( ) | ( ) |
| **소음 환경(청음 난이도)** | ( ) | ( ) | ( ) |
| **특별 고려 사항** | ( ) | ( ) | ( ) |

마. AI 제안 검토 기록

| **AI가 제안한 항목** | **채택 여부** | **채택·미채택 이유** |
| --- | --- | --- |
| **제안 1** |  |  |
| **제안 2** |  |  |
| **제안 3** |  |  |
| **제안 4** |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 사례 카드 6건의 문제점을 모두 역할 정의에 반영했다 | ( ) |
| 7단계 모두 역할 분담 매트릭스 빈칸이 없다 | ( ) |
| 단계마다 '책'이 1명만 지정되었다 | ( ) |
| 현장 체크 항목이 15개 이상이며 모두 눈으로 확인 가능한 문장이다 | ( ) |
| 환경 변수표 3개 부대 유형을 모두 채웠다 | ( ) |
| AI 제안의 채택 여부와 이유를 기록했다 | ( ) |
| 실제 부대명·위치·호출부호를 쓰지 않았다 | ( ) |

**



In [ ]:
# [미션 1 실습 코드] RACI 매트릭스 및 체크리스트 자동 검증
raci_data = {
    '포착': {'관측자': '수·책', '지휘자': '통', '경계조': '협'},
    '식별': {'관측자': '수·책', '지휘자': '통', '경계조': '협'},
    '보고': {'관측자': '수·책', '지휘자': '통', '경계조': '-'},
    '전파': {'관측자': '통', '지휘자': '수·책', '경계조': '통'},
    '결심': {'관측자': '협', '지휘자': '수·책', '경계조': '통'},
    '대피·대응': {'관측자': '협', '지휘자': '책', '경계조': '수'},
    '해제': {'관측자': '통', '지휘자': '수·책', '경계조': '통'}
}

checklist_items = [
    '1. 관측 구역 방위 식별 가능 여부',
    '2. 1차 무전 보고 내 방위각 포함 여부',
    '3. 비행 고도 상대표현(H1~H4) 적용 여부',
    '4. 로터 소음 특성(고주파/저주파) 기록 여부',
    '5. 아군 사전 비행계획 대조 완료 여부',
    '6. 지휘자 복창 확인 3초 내 실행 여부',
    '7. 분대 비상 경보망 즉시 발령 여부',
    '8. 경계조 3초 내 지정 은폐호 진입 여부',
    '9. 인접 진지 유선/무전 상황 전파 여부',
    '10. 대공 사격 통제선 확인 여부',
    '11. 지휘자 구두 사격 승인 확인 여부',
    '12. 적 드론 급강하 궤적 지속 관측 여부',
    '13. 페이로드 낙하물 존재 유무 식별 여부',
    '14. 상황 종료 후 전 대원 이상 유무 확인 여부',
    '15. 배터리 및 통신 잔량 점검 완료 여부'
]

verify_mission1_raci(raci_data, checklist_items)
